In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
import os
import pickle
import pandas as pd
import numpy as np
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random
from itertools import product
from typing import List, Tuple, Optional
from IPython.display import clear_output
%matplotlib inline

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

# desktop version
data_folder = "G:\Meu Drive\mestrado_final_files\data\\"
models_folder = "G:\Meu Drive\mestrado_final_files\models\\NGramModel_ExtraInfo\\"

# google colab version
# data_folder = "drive/MyDrive/mestrado_final_files/data/"
# models_folder = "drive/MyDrive/mestrado_final_files/models/NGramModel_ExtraInfo/"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
pd.set_option("display.max_columns", None)
pd.set_option("display.expand_frame_repr", False)

In [27]:
BATCH_SIZE = 1024
EPOCHS = 50
VOCAB_SIZE = 601
TIME_SIZE = 301
STATE_EMBEDDING_DIM = 16
STATE_SIZE = 225

run_models = 1

In [5]:
%run 01_functions_training.ipynb
#%run drive/MyDrive/mestrado_final_files/notebooks/01_functions_training.ipynb

In [6]:
with open(data_folder + 'data_token_features.pkl', 'rb') as f:
    df_token_features = pickle.load(f)

In [7]:
df_ = df_token_features.query("split == 'train'")

trn_tkn = torch.tensor(np.asarray(df_["token_features"].to_list()), dtype = torch.int64, device = device)
trn_stt = torch.tensor(np.asarray(df_["token_state"].to_list()), dtype = torch.int64, device = device)
trn_y   = torch.tensor(np.asarray(df_["target_tokens"].to_list()), dtype = torch.int64, device = device)

df_ = df_token_features.query("split == 'dev'")

dev_tkn = torch.tensor(np.asarray(df_["token_features"].to_list()), dtype = torch.int64, device = device)
dev_stt = torch.tensor(np.asarray(df_["token_state"].to_list()), dtype = torch.int64, device = device)
dev_y   = torch.tensor(np.asarray(df_["target_tokens"].to_list()), dtype = torch.int64, device = device)

del df_

In [ ]:
list_context = [5]
list_embedding_layer_dim = [32]
list_num_neurons = [50]
list_num_layers = [3]
list_dropout = [0, 0.3]
list_layernorm = [False, True]

In [9]:
model_mask_name = "{}_fulldata_context{}_embedding{}_{}_layernorm{}_setcovariables{}_parameters.pth"

In [31]:
all_combinations = list(product(list_context, list_embedding_layer_dim, list_num_neurons, list_num_layers, list_dropout,
                                list_layernorm))

In [36]:
for comb in all_combinations:
    CONTEXT_SIZE, EMBEDDING_LAYER_SIZE, NUM_HIDDEN_NEURONS, NUM_LAYERS, dropout_rate, layer_norm = comb

    print(comb)

    tp_dropout = "nodropout" if dropout_rate == 0 else "dropout" + str(dropout_rate).replace(".", "")


    token_model = NGramModel_State(
        context_size = CONTEXT_SIZE,
        vocab_size = VOCAB_SIZE,
        embedding_dim = EMBEDDING_LAYER_SIZE,
        state_embedding_dim = STATE_EMBEDDING_DIM,
        state_vocab_size = STATE_SIZE,
        num_hidden_neurons = NUM_HIDDEN_NEURONS,
        num_layers = NUM_LAYERS,
        activation = nn.GELU,
        dropout_rate = dropout_rate,
        use_layer_norm = layer_norm)

    model_filename = model_mask_name\
    .format(token_model.__class__.__name__, CONTEXT_SIZE, EMBEDDING_LAYER_SIZE, tp_dropout, str(layer_norm), 0)

    results_filename = "results_"  + model_filename.replace("_parameters.pth", ".pkl")

    if results_filename in os.listdir(models_folder):
        print("Already fitted.")
        continue

    token_model, trnloss, devloss = train_state_ngram_model(
        model = token_model,
        train_token_data = trn_tkn[:, -CONTEXT_SIZE:],
        train_state_data = trn_stt,
        train_targets = trn_y,
        batch_size = BATCH_SIZE,
        n_epochs = EPOCHS,
        lr = 0.002,
        val_token_data = dev_tkn[:, -CONTEXT_SIZE:],
        val_state_data = dev_stt,
        val_targets = dev_y,
        params_optimizer = {'factor': 0.5, 'threshold': 0, 'patience': 1})

    df_importance = permutation_importance_ngram(
        model = token_model,
        target = dev_y,
        tokens = dev_tkn[:, -CONTEXT_SIZE:],
        state_tokens = dev_stt,
        num_repeats = 10)

    print(df_importance)

    torch.save(token_model.state_dict(), models_folder + model_filename)

    dict_results = {
        "class_model": token_model.__class__.__name__,
        "model_filename": model_filename,
        "token_context_size": CONTEXT_SIZE,
        "token_embedding_dim": EMBEDDING_LAYER_SIZE,
        "token_state_embedding_dim": STATE_EMBEDDING_DIM,
        "state_embedding_vocab_size": STATE_SIZE,
        "training_epochs": EPOCHS,
        "training_batch_loss": np.asarray(trnloss),
        "dev_loss": np.asarray(devloss),
        "num_parameters": get_num_parameters(token_model),
        "variable_importance": df_importance}

    with open(models_folder + results_filename, 'wb') as f:
        pickle.dump(dict_results, f)

    clear_output(wait = True)

(5, 32, 50, 3, 0.3, True)
Baseline model loss:  2.8633406 | Val Loss: 2.86337 | LR: 1.56e-05
       feature  count     mean     std      min      25%      50%      75%      max  baseline     perc   percacm
4      token_1   10.0  10.9617  0.0145  13.8029  13.8145  13.8233  13.8373  13.8436    2.8633  91.0758   91.0758
3      token_2   10.0   0.6846  0.0039   3.5423   3.5461   3.5469   3.5495   3.5559    2.8633   5.6881   96.7639
5  state_token   10.0   0.2292  0.0007   3.0912   3.0923   3.0925   3.0928   3.0936    2.8633   1.9042   98.6681
2      token_3   10.0   0.1377  0.0015   2.9979   3.0002   3.0012   3.0021   3.0028    2.8633   1.1439   99.8120
1      token_4   10.0   0.0180  0.0004   2.8808   2.8810   2.8813   2.8814   2.8820    2.8633   0.1499   99.9620
0      token_5   10.0   0.0046  0.0001   2.8677   2.8678   2.8678   2.8679   2.8679    2.8633   0.0380  100.0000
